In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from ase.io import read
from ase.neighborlist import NeighborList

In [ ]:
atoms = read('coded_JT_smooth_10.xyz', index=':')

len(atoms)

In [ ]:
def get_l_s(atom, cutoff, mn_indices, o_indices, long_cutoff):
    cutoffs = [cutoff/2] * len(atoms[0])
    nl = NeighborList(cutoffs, self_interaction=False, bothways=True, skin=0.0)
    nl.update(atom)

    l = np.full((len(mn_indices)), np.nan)
    for j, mn_index in enumerate(mn_indices):
        indices, offsets = nl.get_neighbors(mn_index)
        for k, (index, offset) in enumerate(zip(indices, offsets)):
            if index in o_indices:
                vec = atom.positions[index] + np.dot(offset, atom.get_cell()) - atom.positions[mn_index]
                length = np.linalg.norm(vec)
                if length > long_cutoff:
                    if -1 > np.dot(vec, [1,0,0]) or np.dot(vec, [1,0,0]) > 1:
                        l[j] = 0
                    if -1 > np.dot(vec, [0,1,0]) or np.dot(vec, [0,1,0]) > 1:
                        l[j] = 1
    return l

In [ ]:
mn_indices = [atom.index for atom in atoms[0] if atom.symbol == 'Mn']
o_indices = [atom.index for atom in atoms[0] if atom.symbol == 'O']

cutoff=3

In [ ]:
l = get_l_s(atoms[0], cutoff, mn_indices, o_indices, 2.05)

In [ ]:
def get_bonds(atoms, cutoff, mn_indices, o_indices, l):

    l_bonds = []
    m_bonds = []
    s_bonds = []

    for atom in atoms:
        cutoffs = [cutoff/2] * len(atoms[0])
        nl = NeighborList(cutoffs, self_interaction=False, bothways=True, skin=0.0)
        nl.update(atom)

        l_bond = []
        m_bond = []
        s_bond = []
        for mn_index, axes in zip(mn_indices, l):
            indices, offsets = nl.get_neighbors(mn_index)
            for index, offset in zip(indices, offsets):
                if index in o_indices:
                    vec = atom.positions[index] + np.dot(offset, atom.get_cell()) - atom.positions[mn_index]
                    if axes == 0:
                        if np.abs(np.dot(vec, [1,0,0])) > 1:
                            l_bond.append([vec, mn_index, index])
                        elif np.abs(np.dot(vec, [0,1,0])) > 1:
                            s_bond.append([vec, mn_index, index])
                    if axes == 1:
                        if np.abs(np.dot(vec, [0,1,0])) > 1:
                            l_bond.append([vec, mn_index, index])
                        elif np.abs(np.dot(vec, [1,0,0])) > 1:
                            s_bond.append([vec, mn_index, index])
                    if np.abs(np.dot(vec, [0,0,1])) > 1:
                        m_bond.append([vec, mn_index, index])

        
        l_bonds.append(l_bond)
        m_bonds.append(m_bond)
        s_bonds.append(s_bond)
                    
    return l_bonds, m_bonds, s_bonds

In [ ]:
l_bonds, m_bonds, s_bonds = get_bonds(atoms, cutoff, mn_indices, o_indices, l)

In [ ]:
l_bond_lengths = [[np.linalg.norm(vec[0]) for vec in bond] for bond in l_bonds]
m_bond_lengths = [[np.linalg.norm(vec[0]) for vec in bond] for bond in m_bonds]
s_bond_lengths = [[np.linalg.norm(vec[0]) for vec in bond] for bond in s_bonds]

In [ ]:
l_means = [np.mean(bond) for bond in l_bond_lengths]
m_means = [np.mean(bond) for bond in m_bond_lengths]
s_means = [np.mean(bond) for bond in s_bond_lengths]

In [ ]:
l_std = [np.std(bond) for bond in l_bond_lengths]
m_std = [np.std(bond) for bond in m_bond_lengths]
s_std = [np.std(bond) for bond in s_bond_lengths]

In [ ]:
x_1 = [323.91089082914806, 372.7008359992085, 581.8005440386042, 592.255529440574, 603.2913650904774, 613.1655002445136, 621.877974785287, 633.4946606831238, 643.3687958371602, 652.6620807432628, 661.955445414574, 671.8295805686103, 681.7037157226465, 692.1587011246163, 700.8712155479939, 712.4879014458309, 722.362036599867, 730.4937007753111, 742.1103069079395, 753.7269928057765, 762.4394274639455, 771.7327921352568, 782.7686277851601, 790.9002919606041, 803.0977483411662, 812.9718834952024, 821.6843979185801, 831.5585330726162, 843.1752189704532, 851.8876536286223, 861.7617887826586, 872.2167741846283, 883.2526895997402, 892.5459745058429, 902.4201096598791, 912.2943245791238, 923.9110104769607, 933.204215617855, 942.4975802891661, 950.6291646994018, 962.2458505972386, 974.4433069778006, 983.1558214011782, 991.2874855766223]
y_1 = [2.2019059712309073, 2.1980940222254577, 2.18132147408475, 2.182465052897113, 2.178271917497845, 2.177128319054576, 2.1775095250793006, 2.175603553848394, 2.174459975036031, 2.172554003805124, 2.170648026030582, 2.169504434130948, 2.169504434130948, 2.167979668924766, 2.1641677264629524, 2.1614993628135015, 2.1557814491207807, 2.1534942784087843, 2.1477763647160635, 2.105844984548841, 2.057433298270354, 2.049046988210005, 2.041423116373648, 2.0376111608245635, 2.0353240293743795, 2.0349428233496543, 2.0311308678005697, 2.031893253675478, 2.031512073825295, 2.028843710175844, 2.030749687950386, 2.028843710175844, 2.029224916200569, 2.028843710175844, 2.0280812981263936, 2.0280812981263936, 2.02770011827621, 2.0296060960507525, 2.028843710175844, 2.0284625303256605, 2.0280812981263936, 2.027318938426027, 2.026556552551118, 2.0242693687518507]
x_2 = [321.5876094852266, 371.5391753859456, 582.9622046518671, 592.8363796885076, 602.1296645946103, 610.2613287700544, 621.877974785287, 631.7521099393232, 641.0454746106344, 650.3387595167371, 663.1170661452327, 672.4104308165438, 683.4462664664471, 691.5779306418913, 700.8712155479939, 711.9071309631057, 722.362036599867, 732.8170220018369, 743.8528576517401, 753.7269928057765, 762.4394274639455, 772.3136423831903, 782.1877775372266, 792.0619126912628, 801.355277362574, 812.3910332472689, 822.8460186492387, 831.5585330726162, 842.0135184745861, 852.4685038765558, 862.3426390305922, 873.3784746804954, 880.929288608006, 892.5459745058429, 903.5818101557462, 912.8750950618489, 923.3300804638187, 933.204215617855, 943.0783507718912, 952.3717154432023, 963.4075510931057, 972.1199857512748, 980.8325001746525, 993.0299565552144]
y_2 = [1.9720457400115494, 1.9747140774864587, 1.980813216834811, 1.980813216834811, 1.980813216834811, 1.9819568087344446, 1.9823379624100863, 1.9842439401846286, 1.980813216834811, 1.9838627865089868, 1.9834815804842618, 1.9842439401846286, 1.9846251723838955, 1.9850063522340788, 1.9857687381089875, 1.9838627865089868, 1.9872935360333464, 1.989199487633347, 1.9876747158835297, 1.9979669644566083, 2.0010165341307844, 2.004066077630419, 2.0033036655809684, 2.0033036655809684, 2.0033036655809684, 2.004066077630419, 2.004066077630419, 2.004066077630419, 2.0067344412798698, 2.0090215727300538, 2.0067344412798698, 2.0090215727300538, 2.0090215727300538, 2.010165190804229, 2.0105463706544127, 2.0120711685787716, 2.0132147343038636, 2.013977120178772, 2.013595940328589, 2.013977120178772, 2.014739506053681, 2.0158830979533144, 2.017026689852948, 2.0189326676274906]
x_3 = [320.4259089893595, 371.5391753859456, 582.9622046518671, 590.5130185793777, 602.7105148425438, 612.0038396312507, 621.877974785287, 632.9138104351903, 642.7879455892266, 650.3387595167371, 662.5362956625075, 671.8295805686103, 682.28456597058, 690.9970803939577, 700.8712155479939, 713.6495221764894, 722.362036599867, 729.9128505273776, 742.691157155873, 752.5653720751178, 761.2778864984955, 768.247770412864, 772.8944926311239, 782.1877775372266, 792.0619126912628, 802.5168980932326, 812.9718834952024, 824.0077191451057, 832.7202335684833, 842.0135184745861, 853.0493541244894, 863.5043395264593, 873.9593249284289, 882.0909891038731, 891.9652040231178, 901.8392594119456, 912.8750950618489, 923.3300804638187, 931.4617446392629, 941.9167300412325, 951.7908651952689, 962.2458505972386, 972.1199857512748, 982.5749711532446, 993.6108865683565]
y_3 = [1.891994935226189, 1.8935197069760061, 1.904193135399268, 1.907242705073444, 1.905717933323627, 1.9064803191985356, 1.907242705073444, 1.908005064773811, 1.908005064773811, 1.9095298626981698, 1.9095298626981698, 1.9102922485730784, 1.9095298626981698, 1.9102922485730784, 1.9121982263476207, 1.9152477960217968, 1.918297339521431, 1.9198221112712484, 1.924777632545425, 1.9434561780915813, 1.9789072390602684, 1.9838627865089868, 1.9869123300086213, 1.993392623032615, 1.9968233725569746, 1.9994917100318839, 2.0013976878064264, 2.001778920005693, 2.0021600998558764, 2.0029224857307852, 2.0033036655809684, 2.0033036655809684, 2.0036848716056936, 2.004066077630419, 2.0029224857307852, 2.0013976878064264, 2.0010165341307844, 2.000635328106059, 1.9983481704813335, 2.000254122081334, 1.9991105563562421, 1.9983481704813335, 1.9983481704813335, 1.9987293503315169, 1.9998729422311508]

In [ ]:
steps = np.linspace(200,1200,len(l_means))
plt.plot(steps, l_means, color='black')
plt.plot(steps, m_means, color='black')
plt.plot(steps, s_means, color='black')
plt.fill_between(steps, np.array(l_means) - np.array(l_std), np.array(l_means) + np.array(l_std), alpha=0.2, color='black', linewidth=0.01)
plt.fill_between(steps, np.array(m_means) - np.array(m_std), np.array(m_means) + np.array(m_std), alpha=0.2, color='black', linewidth=0.01)
plt.fill_between(steps, np.array(s_means) - np.array(s_std), np.array(s_means) + np.array(s_std), alpha=0.2, color='black', linewidth=0.01)

plt.xlim(300, 1000)
plt.ylim(1.85, 2.3)
plt.xlabel('Temperature (K)')
plt.ylabel(r'Mn-O bond length ($\AA$)')
plt.yticks([1.9, 2.0, 2.1, 2.2])
plt.xticks([400, 600, 800, 1000])
plt.savefig("bond_evolution_smooth_v3.png")
plt.show()

In [ ]:
plt.plot(x_1, y_1, marker='o', color='red', label='Experiment')
plt.plot(x_2, y_2, marker='o', color='red')
plt.plot(x_3, y_3, marker='o', color='red')

plt.ylim(1.85, 2.3)
plt.xlim(300, 1000)
plt.xlabel('Temperature (K)')
plt.ylabel(r'Mn-O bond length ($\AA$)')
plt.yticks([1.9, 2.0, 2.1, 2.2])
plt.xticks([400, 600, 800, 1000])
plt.legend(frameon=False)
plt.savefig("bond_evolution_Experiment_v3.png")
plt.show()